# Process the Power Curve from the thewindpower.net.
This script demonstrates how to use the power curves from www.thewindpower.net within RESKit. To run this example, you need to purchase the 'Power_curves_yyyymmdd.xls' power curves and the 'Turbines_yyyymmdd.xls' turbine specification data from www.thewindpower.net.

The turbine specification data is mainly used to determine the power rating of the turbines corresponding to the power curves.



In [2]:
import reskit as rk
import pandas as pd
import numpy as np
import shutil
import pathlib

In [3]:
# Define path to the power curves and turbine library from thewindpower.net

# TODO Adjust the path to the location of your purcharsed files
path_power_curves = r"Power_curves_20230708.xls"
path_turbine_library = r"Turbines_20230708.xls"


output_folder = "..."
# specify the source of the data for information purposes
source = "World Wind Farms 2023"

In [4]:
df_power_curves = pd.read_excel(path_power_curves, sheet_name="Power_curves", header=[0, 1])

df_turbine_library = pd.read_excel(path_turbine_library, sheet_name="Turbines", skiprows=[1])
df_turbine_library["Name"] = df_turbine_library["Name"].astype(str)
df_turbine_library["Manufacturer"] = df_turbine_library["Manufucturer"].astype(str)  # also rename typo

template_dict = {
    "Manufacturer": None,
    "Model": None,
    "Capacity": None,
    "Usage": None,
    "HubHeight": None,
    "Source": source,
    "RotorDiameter": None,
    "Power curve": None,
    "windspeed(m/s)": "power(kW)",
}


ws_list = df_power_curves.columns.levels[1].tolist()[0:71]

FileNotFoundError: [Errno 2] No such file or directory: 'Power_curves_20230708.xls'

In [ ]:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for i, row in df_power_curves.iterrows():
    single_turbine_df = pd.DataFrame.from_dict(template_dict, orient="index").squeeze()

    # turbine_lib
    turbine_id = row["Turb. ID"].values[0]
    turbine_lib_info = df_turbine_library.loc[df_turbine_library["ID"] == turbine_id].squeeze()

    single_turbine_df.loc["Manufacturer"] = row["Manufucturer Name"].values[0]
    manufacturer = row["Manufucturer Name"].values[0].replace(" ", "_")
    model_TWP = row["Turbine Name"].astype(str).values[0]
    model_RESKit = model_TWP.replace("/", "-").replace(" ", "_")

    # manufacturer is added to avoid duplicates
    model_RESKit = f"{model_RESKit}_{manufacturer}"

    single_turbine_df.loc["Model"] = model_RESKit
    single_turbine_df.loc["Capacity"] = turbine_lib_info["Rated power"]
    if turbine_lib_info["Offshore"] == "No":
        onshore_offshore = "Onshore"
    elif turbine_lib_info["Offshore"] == "Yes":
        onshore_offshore = "Offshore"
    else:
        onshore_offshore = None

    single_turbine_df.loc["Usage"] = onshore_offshore
    min_hub_height = turbine_lib_info["Minimum hub height"]
    max_hub_height = turbine_lib_info["Maximum hub height"]

    if min_hub_height == "#ND":
        min_hub_height = np.nan
        max_hub_height = np.nan

    single_turbine_df.loc["HubHeight"] = f"{min_hub_height}, {max_hub_height}"
    single_turbine_df.loc["RotorDiameter"] = float(turbine_lib_info["Rotor diameter"])

    # read power curve
    pc_values = row[4:75].values
    pc_df_turbine = pd.Series(pc_values, index=ws_list)

    single_turbine_df = pd.concat([single_turbine_df, pc_df_turbine], axis=0)

    single_turbine_df.to_csv(f"{output_folder}/{model_RESKit}.csv", index=True, header=False)

# Set new path to the turbine library

Afterwards set __turbine_library_path__ in default_paths.yaml (located inside reskit folder) to the path of __output_folder__